In [ ]:
import random
from collections import defaultdict

In [ ]:
# 공통 설정
N = 4
START = (0, 0)
GOAL = (3, 3)
ACTIONS = [0, 1, 2, 3]  # 0: left, 1: up, 2: right, 3: down

GAMMA = 0.9
ALPHA = 0.1
EPISODES = 1000

In [ ]:
# 기본 4x4 GridWorld (벽 없음)
class BasicGridWorld:
    def __init__(self):
        self.reset()

    def reset(self):
        self.agent_pos = START
        return self.agent_pos

    def step(self, action):
        x, y = self.agent_pos

        if action == 0:      # left
            y = max(0, y - 1)
        elif action == 1:    # up
            x = max(0, x - 1)
        elif action == 2:    # right
            y = min(N - 1, y + 1)
        elif action == 3:    # down
            x = min(N - 1, x + 1)

        self.agent_pos = (x, y)

        # 보상 및 종료
        if self.agent_pos == GOAL:
            return self.agent_pos, 0.0, True  # 보상을 0으로 두고 싶다면 이렇게
        else:
            return self.agent_pos, -1.0, False

In [ ]:
# 랜덤 벽 GridWorld
class RandomWallGridWorld:
    def __init__(self):
        self.walls = set()
        self.reset()

    def _generate_walls(self):
        self.walls.clear()
        num_walls = random.choice([2, 3])
        candidates = [(i, j) for i in range(N) for j in range(N)
                      if (i, j) not in (START, GOAL)]
        random.shuffle(candidates)
        for pos in candidates:
            if len(self.walls) >= num_walls:
                break
            self.walls.add(pos)

    def reset(self):
        # 에피소드마다 새로운 벽 생성
        self._generate_walls()
        self.agent_pos = START
        return self.agent_pos

    def step(self, action):
        x, y = self.agent_pos
        nx, ny = x, y

        if action == 0:      # left
            ny = max(0, y - 1)
        elif action == 1:    # up
            nx = max(0, x - 1)
        elif action == 2:    # right
            ny = min(N - 1, y + 1)
        elif action == 3:    # down
            nx = min(N - 1, x + 1)

        # 격자 밖은 이미 막았고, 벽이면 제자리 유지
        if (nx, ny) in self.walls:
            nx, ny = x, y

        self.agent_pos = (nx, ny)

        if self.agent_pos == GOAL:
            return self.agent_pos, 0.0, True
        else:
            return self.agent_pos, -1.0, False

In [ ]:
# TD(0) 학습 공통 루프
def td_learn(env, episodes=EPISODES, alpha=ALPHA, gamma=GAMMA,
             max_steps=100, name=""):
    V = defaultdict(float)
    for ep in range(episodes):
        s = env.reset()
        done = False

        for t in range(max_steps):
            if done:
                break
            a = random.choice(ACTIONS)
            s_prime, r, done = env.step(a)

            old_v = V[s]
            target = r + (gamma * V[s_prime] if not done else r)
            V[s] = old_v + alpha * (target - old_v)

            s = s_prime

        if (ep + 1) % 100 == 0 and name:
            print(f"[{name}] episode {ep+1}/{episodes}")
    return V

def print_value_table(V, title):
    print(title)
    for i in range(N):
        row = []
        for j in range(N):
            v = V[(i, j)]
            row.append(f"{v:6.2f}")
        print(" ".join(row))
    print()

In [ ]:
if __name__ == "__main__":
    random.seed(2026)

    # 기본 GridWorld 학습
    basic_env = BasicGridWorld()
    V_basic = td_learn(basic_env, name="basic")
    print_value_table(V_basic, "[기본 4x4 GridWorld 상태가치 V(s)]")

    # 랜덤 벽 GridWorld 학습
    wall_env = RandomWallGridWorld()
    V_wall  = td_learn(wall_env,  name="wall")
    print_value_table(V_wall, "[랜덤 벽 GridWorld 상태가치 V(s)]")

[basic] episode 100/1000
[basic] episode 200/1000
[basic] episode 300/1000
[basic] episode 400/1000
[basic] episode 500/1000
[basic] episode 600/1000
[basic] episode 700/1000
[basic] episode 800/1000
[basic] episode 900/1000
[basic] episode 1000/1000
[기본 4x4 GridWorld 상태가치 V(s)]
 -9.33  -9.12  -8.79  -8.46
 -9.20  -8.91  -8.56  -7.35
 -8.81  -8.38  -7.10  -4.28
 -8.62  -7.64  -4.59   0.00

[wall] episode 100/1000
[wall] episode 200/1000
[wall] episode 300/1000
[wall] episode 400/1000
[wall] episode 500/1000
[wall] episode 600/1000
[wall] episode 700/1000
[wall] episode 800/1000
[wall] episode 900/1000
[wall] episode 1000/1000
[랜덤 벽 GridWorld 상태가치 V(s)]
 -9.51  -9.41  -9.09  -8.81
 -9.39  -9.21  -8.59  -7.40
 -9.19  -9.01  -7.34  -4.66
 -9.07  -8.55  -5.23   0.00

